# HOUSE PRICES

## Setup And Imports

In [40]:
import numpy as np
import pandas as pd
import mlflow
import matplotlib.pyplot as plt
import dagshub
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import SelectKBest, f_regression

import xgboost as xgb
import lightgbm as lgb
from scipy.stats import skew

dagshub.init(repo_owner='LukaBatilashvili07', repo_name='house-prices', mlflow=True)
mlflow.set_experiment('house-prices-experiment')

RANDOM_STATE = 42

Initialized MLflow to track repo "LukaBatilashvili07/house-prices"

Repository LukaBatilashvili07/house-prices initialized!

## Data Loading

In [41]:
train_data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

print('Train shape:', train_data.shape)
print('Test shape:', test_data.shape)
train_data.head()

Train shape: (1460, 81)
Test shape: (1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [42]:
missing_values = train_data.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)
print("Missing values in training data:")
print(missing_values)

Missing values in training data:
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtFinType1      37
BsmtCond          37
BsmtQual          37
MasVnrArea         8
Electrical         1
dtype: int64


## Data Cleaning

In [43]:
test_ids = test_data['Id']
all_data = pd.concat([train_data.drop('SalePrice', axis=1), test_data], axis=0).reset_index(drop=True)

none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 
            'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
            'MasVnrType']
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
                'BsmtFullBath', 'BsmtHalfBath']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType',
            'Functional', 'Utilities']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

print("Missing values after imputation:")
print(all_data.isnull().sum()[all_data.isnull().sum() > 0])

Missing values after imputation:
Series([], dtype: int64)


In [44]:
train_cleaned = train_data.copy()
outliers = train_cleaned[(train_cleaned['GrLivArea'] > 4000) & (train_cleaned['SalePrice'] < 300000)].index
train_cleaned = train_cleaned.drop(outliers)

y = np.log1p(train_cleaned['SalePrice'])
print('Target shape', y.shape)

Target shape (1458,)


## Feature Engineering

In [45]:
def feature_engineering(df):
    df = df.copy()
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['TotalPorchSF'] = df['OpenPorchSF'] + df['EnclosedPorch'] + df['3SsnPorch'] + df['ScreenPorch']
    df['Age'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
    df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['IsRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

    qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
    qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
    for col in qual_cols:
        if col in df.columns:
            df[col] = df[col].map(qual_mapping).fillna(0)

    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df[col] = df[col].astype('category').cat.codes
    return df

all_data_fe = feature_engineering(all_data)
print('Data shape after feature engineering:', all_data_fe.shape)

Data shape after feature engineering: (2919, 90)


In [46]:
len_train = len(train_cleaned)
X_full = all_data_fe[:len_train].copy()
X_test = all_data_fe[len_train:].copy()

X_train, X_val, y_train, y_val = train_test_split(X_full, y, test_size=0.2, random_state=RANDOM_STATE)

numeric_features = X_train.select_dtypes(include=[np.number]).columns
skewed = X_train[numeric_features].apply(lambda x: skew(x.dropna()))
skewed_features = skewed[skewed > 0.75].index
X_train[skewed_features] = np.log1p(X_train[skewed_features])
X_val[skewed_features] = np.log1p(X_val[skewed_features])
X_test[skewed_features] = np.log1p(X_test[skewed_features])

## Feature Selection

In [47]:
rf_selector = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_selector.fit(X_train, y_train)
rf_importances = pd.Series(rf_selector.feature_importances_, index=X_train.columns)
rf_selected_features = rf_importances[rf_importances > 0.01].index
print(len(rf_selected_features), "features selected")

X_train = X_train[rf_selected_features]
X_val = X_val[rf_selected_features]
X_test_selected = X_test[rf_selected_features]

28 features selected


## Model Training And MLFlow Logging

In [48]:
def evaluate_model(model, X_train, y_train, X_val, y_val):
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    train_r2 = r2_score(y_train, train_preds)
    val_r2 = r2_score(y_val, val_preds)
    return train_rmse, val_rmse, train_r2, val_r2

results = {}

In [49]:
with mlflow.start_run(run_name='Linear Regression'):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lr, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.sklearn.log_model(lr, 'linear_regression_model')
    results['Linear Regression'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'Linear Regression - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:48:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:48:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Linear Regression - Train RMSE: 0.3622, Val RMSE: 0.3962, Train R2: 0.1667, Val R2: 0.0689
🏃 View run Linear Regression at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/f2c9d2aa441842e18379b09b1a19d5fb
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [50]:
for alpha in [0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f'Lasso alpha={alpha}'):
        lasso = Lasso(alpha=alpha, random_state=RANDOM_STATE)
        lasso.fit(X_train, y_train)
        train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lasso, X_train, y_train, X_val, y_val)
        mlflow.log_metric('train_rmse', train_rmse)
        mlflow.log_metric('val_rmse', val_rmse)
        mlflow.log_metric('train_r2', train_r2)
        mlflow.log_metric('val_r2', val_r2)
        mlflow.sklearn.log_model(lasso, 'lasso_model')
        results[f'Lasso alpha={alpha}'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
        print(f'Lasso alpha={alpha} - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:49:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:49:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso alpha=0.1 - Train RMSE: 0.3784, Val RMSE: 0.4024, Train R2: 0.0906, Val R2: 0.0393
🏃 View run Lasso alpha=0.1 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/f77971baf2104d13b2ecf207e2733750
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


2026/04/14 13:49:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:49:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso alpha=1.0 - Train RMSE: 0.3818, Val RMSE: 0.4054, Train R2: 0.0741, Val R2: 0.0253
🏃 View run Lasso alpha=1.0 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/a9b478bbc1814045a7cd5ca5d758ec7b
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


2026/04/14 13:49:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:50:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Lasso alpha=10.0 - Train RMSE: 0.3861, Val RMSE: 0.4060, Train R2: 0.0529, Val R2: 0.0220
🏃 View run Lasso alpha=10.0 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/dafcaa76570949d4b9354d02825da894
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [51]:
for alpha in [0.1, 1.0, 10.0]:
    with mlflow.start_run(run_name=f'Ridge alpha={alpha}'):
        ridge = Ridge(alpha=alpha)
        ridge.fit(X_train, y_train)
        train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(ridge, X_train, y_train, X_val, y_val)
        mlflow.log_metric('train_rmse', train_rmse)
        mlflow.log_metric('val_rmse', val_rmse)
        mlflow.log_metric('train_r2', train_r2)
        mlflow.log_metric('val_r2', val_r2)
        mlflow.sklearn.log_model(ridge, 'ridge_model')
        results[f'Ridge alpha={alpha}'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
        print(f'Ridge alpha={alpha} - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:50:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:50:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge alpha=0.1 - Train RMSE: 0.3622, Val RMSE: 0.3962, Train R2: 0.1666, Val R2: 0.0687
🏃 View run Ridge alpha=0.1 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/ea65c97b88c34efa8c1c6b9891afc1a3
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


2026/04/14 13:50:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:50:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge alpha=1.0 - Train RMSE: 0.3624, Val RMSE: 0.3967, Train R2: 0.1657, Val R2: 0.0667
🏃 View run Ridge alpha=1.0 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/d3b8b1949ffd486f95199e680600b944
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


2026/04/14 13:51:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:51:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Ridge alpha=10.0 - Train RMSE: 0.3637, Val RMSE: 0.3980, Train R2: 0.1597, Val R2: 0.0603
🏃 View run Ridge alpha=10.0 at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/16525849d9a140d5938b6acbdf00f392
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [52]:
with mlflow.start_run(run_name='Random Forest'):
    rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    rf.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(rf, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.sklearn.log_model(rf, 'random_forest_model')
    results['Random Forest'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'Random Forest - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:51:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:51:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest - Train RMSE: 0.1286, Val RMSE: 0.3603, Train R2: 0.8949, Val R2: 0.2301
🏃 View run Random Forest at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/87f410afa31b430bad86c1e1ebc51f25
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [53]:
with mlflow.start_run(run_name='XGBoost'):
    xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE)
    xgb_model.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(xgb_model, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.xgboost.log_model(xgb_model, 'xgboost_model')
    results['XGBoost'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'XGBoost - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:51:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost - Train RMSE: 0.1606, Val RMSE: 0.3777, Train R2: 0.8361, Val R2: 0.1536
🏃 View run XGBoost at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/1938d4b6fb3a49679b666aa208d81d3c
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [54]:
with mlflow.start_run(run_name='LightGBM'):
    lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE)
    lgb_model.fit(X_train, y_train)
    train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(lgb_model, X_train, y_train, X_val, y_val)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.lightgbm.log_model(lgb_model, 'lightgbm_model')
    results['LightGBM'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'LightGBM - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000751 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3516
[LightGBM] [Info] Number of data points in the train set: 1166, number of used features: 28
[LightGBM] [Info] Start training from score 12.023362
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

2026/04/14 13:52:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/14 13:52:24 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM - Train RMSE: 0.0961, Val RMSE: 0.4022, Train R2: 0.9414, Val R2: 0.0402
🏃 View run LightGBM at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/2a2cb49f764f43c59d1bfbf497c094df
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1


In [55]:
parameter_grid = {
    'n_estimators': [300, 500],
    'learning_rate': [0.03, 0.05],
    'max_depth': [3, 4],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = xgb.XGBRegressor(random_state=RANDOM_STATE)
xgb_random = RandomizedSearchCV(xgb_base, parameter_grid, n_iter=10, scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE, cv=3)

xgb_random.fit(X_train, y_train)

best_params = xgb_random.best_params_
best_model = xgb_random.best_estimator_
train_rmse, val_rmse, train_r2, val_r2 = evaluate_model(best_model, X_train, y_train, X_val, y_val)

with mlflow.start_run(run_name='XGBoost Hyperparameter Tuning'):
    mlflow.log_params(best_params)
    mlflow.log_metric('train_rmse', train_rmse)
    mlflow.log_metric('val_rmse', val_rmse)
    mlflow.log_metric('train_r2', train_r2)
    mlflow.log_metric('val_r2', val_r2)
    mlflow.xgboost.log_model(best_model, 'model', registered_model_name='best_house_price_model')
    results['XGBoost Tuned'] = {'train_rmse': train_rmse, 'val_rmse': val_rmse, 'train_r2': train_r2, 'val_r2': val_r2}
    print(f'XGBoost Tuned - Train RMSE: {train_rmse:.4f}, Val RMSE: {val_rmse:.4f}, Train R2: {train_r2:.4f}, Val R2: {val_r2:.4f}')

2026/04/14 13:52:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'best_house_price_model' already exists. Creating a new version of this model...
2026/04/14 13:52:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_house_price_model, version 10
Created version '10' of model 'best_house_price_model'.


XGBoost Tuned - Train RMSE: 0.2635, Val RMSE: 0.3705, Train R2: 0.5589, Val R2: 0.1855
🏃 View run XGBoost Hyperparameter Tuning at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1/runs/c08b84e751014af4ba1778ce25aaf8c5
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/house-prices.mlflow/#/experiments/1
